In [1]:
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Text

In [2]:
def parse_rate(rate_text: str) -> float:
    """
    Parse Scritch-style rate strings.

    Examples:
        "4:1"  -> 4.0 steps/beat
        "32:1" -> 32.0 steps/beat
        "3:2"  -> 1.5 steps/beat
        "1:32" -> 0.03125 steps/beat
    """
    text = rate_text.strip()

    match = re.fullmatch(r"(\d+(?:\.\d+)?)\s*:\s*(\d+(?:\.\d+)?)", text)
    if not match:
        raise ValueError(f'Rate must look like "4:1", "3:2", "1:32", etc. Got: {rate_text!r}')

    steps = float(match.group(1))
    beats = float(match.group(2))

    if steps <= 0 or beats <= 0:
        raise ValueError("Rate values must be greater than zero.")

    if steps > 32 or beats > 32:
        raise ValueError("Scritch rates should be in the range 1–32 steps per 1–32 beats.")

    return steps / beats

In [3]:
def curve_exponent(curve: float) -> float:
    """
    Map CURVE -100..+100 to a power-curve exponent.

    curve = 0      -> exponent 1.0, linear
    curve > 0      -> exponent < 1.0, log-ish / changes early
    curve < 0      -> exponent > 1.0, exp-ish / changes late
    """
    return 2 ** (-curve / 50.0)


def shaped_progress(x: np.ndarray | float, curve: float):
    a = curve_exponent(curve)
    return np.power(x, a)


def rate_at(t, duration: float, start_rate: float, end_rate: float, curve: float):
    x = np.asarray(t) / duration
    g = shaped_progress(x, curve)
    return start_rate + (end_rate - start_rate) * g

In [4]:
def accumulated_area(t, duration: float, start_rate: float, end_rate: float, curve: float):
    """
    Closed form for:

        R(t) = R0 + (R1 - R0) * (t / D)^a

    A(t) = integral of R(t) from 0 to t.
    """
    t = np.asarray(t)
    a = curve_exponent(curve)
    x = t / duration

    return (
        start_rate * t
        + (end_rate - start_rate) * duration * np.power(x, a + 1) / (a + 1)
    )

In [ ]:
def find_time_for_area(k: int, duration: float, start_rate: float, end_rate: float, curve: float) -> float:
    """
    Find t where accumulated_area(t) == k.
    Uses binary search because A(t) is monotonic when rate is positive.
    """
    low = 0.0
    high = duration

    for _ in range(80):
        mid = (low + high) / 2
        area = float(accumulated_area(mid, duration, start_rate, end_rate, curve))

        if area < k:
            low = mid
        else:
            high = mid

    return high


def ramp_steps(
    duration: float,
    start_rate: float,
    end_rate: float,
    curve: float,
):
    total_area = float(accumulated_area(duration, duration, start_rate, end_rate, curve))

    epsilon = 1e-9

    # Integer crossings that occur strictly before the terminal endpoint.
    # Step k=0 is the beat-zero anchor.
    max_k = math.floor(total_area - epsilon)

    positions = [
        find_time_for_area(k, duration, start_rate, end_rate, curve)
        for k in range(max_k + 1)
    ]

    intervals = np.diff(positions)

    return {
        "total_area": total_area,
        "step_count": len(positions),
        "positions": np.array(positions),
        "intervals": intervals,
    }

In [6]:
def plot_ramp(duration=4.0, start_rate=3.0, end_rate=4.0, curve=0.0):
    t = np.linspace(0, duration, 1000)
    r = rate_at(t, duration, start_rate, end_rate, curve)
    a = accumulated_area(t, duration, start_rate, end_rate, curve)

    result = ramp_steps(duration, start_rate, end_rate, curve)
    positions = result["positions"]

    plt.figure(figsize=(10, 4))
    plt.plot(t, r)
    plt.title("Instantaneous rate R(t)")
    plt.xlabel("Beat")
    plt.ylabel("Steps per beat")
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(t, a)
    for k in range(result["step_count"]):
        plt.axhline(k, linewidth=0.5, alpha=0.25)
    plt.title("Accumulated area A(t)")
    plt.xlabel("Beat")
    plt.ylabel("Accumulated steps")
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(10, 1.8))
    plt.eventplot(positions, lineoffsets=1, linelengths=0.7)
    plt.xlim(0, duration)
    plt.yticks([])
    plt.title(f"Generated step positions — {result['step_count']} steps")
    plt.xlabel("Beat")
    plt.grid(True)
    plt.show()

In [7]:
def plot_ramp_from_rate_strings(
    duration=4.0,
    start_rate_text="3:1",
    end_rate_text="4:1",
    curve=0.0,
):
    try:
        start_rate = parse_rate(start_rate_text)
        end_rate = parse_rate(end_rate_text)
    except ValueError as error:
        print(f"Invalid rate: {error}")
        return

    result = ramp_steps(duration, start_rate, end_rate, curve)

    print(f"Start rate: {start_rate_text} = {start_rate:.6g} steps/beat")
    print(f"End rate:   {end_rate_text} = {end_rate:.6g} steps/beat")
    print(f"Duration:   {duration:g} beats")
    print(f"Curve:      {curve:g}")
    print(f"Area:       {result['total_area']:.6f}")
    print(f"Step total: {result['step_count']}")

    plot_ramp(
        duration=duration,
        start_rate=start_rate,
        end_rate=end_rate,
        curve=curve,
    )

In [8]:
interact(
    plot_ramp,
    duration=FloatSlider(value=4.0, min=1.0, max=16.0, step=0.25),
    start_rate=FloatSlider(value=3.0, min=1.0, max=32.0, step=0.25),
    end_rate=FloatSlider(value=4.0, min=1.0, max=32.0, step=0.25),
    curve=FloatSlider(value=0.0, min=-100.0, max=100.0, step=1.0),
)

interactive(children=(FloatSlider(value=4.0, description='duration', max=16.0, min=1.0, step=0.25), FloatSlide…

<function __main__.plot_ramp(duration=4.0, start_rate=3.0, end_rate=4.0, curve=0.0)>